# RAG Check — ФКН ВШЭ

**Что делает этот notebook:**
1. Устанавливает зависимости
2. Строит ChromaDB-индекс из `.md` файлов базы знаний
3. Прогоняет каждый вопрос из тест-файла через RAG
4. Считает метрики: context recall, ROUGE-1, ROUGE-L

**Перед запуском:**
- Добавь датасет с `.md` файлами через `+ Add Data`
- Добавь секрет `GROQ_API_KEY` в `Add-ons → Secrets` (если нужна генерация)
- Укажи имя датасета в ячейке **Настройки**

In [ ]:
# Ячейка 1 — установка зависимостей (~3-5 минут первый раз)
!pip install chromadb sentence-transformers groq rouge-score --quiet

In [ ]:
# Ячейка 2 — НАСТРОЙКИ (измени здесь)
import os
from pathlib import Path

# Имя датасета — то что стоит после /kaggle/input/
# Посмотри в боковой панели Data после добавления датасета
DATASET_NAME = "fkn-rag-files"   # <-- поменяй на своё

DOCS_DIR  = Path(f"/kaggle/input/{DATASET_NAME}")
QA_FILE   = DOCS_DIR / "fkn_qa_test (1).md"   # тестовый файл

# Модель Groq для генерации ответов
GROQ_MODEL = "llama-3.3-70b-versatile"

# Сколько чанков достаём из ChromaDB
RAG_TOP_K = 6

# True  — генерировать ответы через Groq (нужен API-ключ)
# False — только retrieval-метрики (быстро, без ключа)
WITH_LLM = True

# Лимит вопросов (0 = все)
LIMIT = 0

# Groq API ключ (через Kaggle Secrets — безопасно)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    print("✅ GROQ_API_KEY загружен из Secrets")
except Exception:
    print("⚠️  GROQ_API_KEY не найден в Secrets — генерация будет отключена")
    WITH_LLM = False

print(f"DOCS_DIR : {DOCS_DIR}")
print(f"QA_FILE  : {QA_FILE}")
print(f"WITH_LLM : {WITH_LLM}")

In [ ]:
# Ячейка 3 — Чанкование .md файлов
import re

_DOCS_EXCLUDE = ("qa_test", "_test", "test_")

def _is_qa_format(text):
    bold_lines = sum(1 for ln in text.splitlines() if re.match(r'^\*\*.+\*\*\s*$', ln))
    return bold_lines >= 5

def _split_qa_chunks(text, source):
    chunks, section, question, answer_lines = [], "", "", []
    def _flush():
        if not question or not answer_lines:
            return
        answer = "\n".join(answer_lines).strip()
        if not answer or "не найден" in answer.lower() or len(answer) < 15:
            return
        q_clean = re.sub(r"\*+", "", question).strip().rstrip("?")
        chunks.append({"text": f"{section}\nВопрос: {q_clean}\nОтвет: {answer}",
                       "heading": q_clean[:80], "source": source})
    for line in text.splitlines():
        if line.startswith("## "):
            _flush(); section = line; question = ""; answer_lines = []
        elif re.match(r'^\*\*.+\*\*\s*$', line):
            _flush(); question = line; answer_lines = []
        elif line.startswith("# ") or line.startswith("---") or line.startswith("*Источник"):
            pass
        elif question and line.strip():
            answer_lines.append(line.strip())
    _flush()
    return chunks

def _split_with_subheadings(text, source):
    chunks, h1_context, h2_heading, sub_heading, current_lines = [], "", "", "", []
    def _flush():
        if not current_lines: return
        content = "\n".join(current_lines).strip()
        if len(content) <= 50: return
        heading_label = sub_heading or h2_heading
        ctx_parts = []
        if h1_context: ctx_parts.append(h1_context)
        if sub_heading and h2_heading: ctx_parts.append(h2_heading)
        prefix = "\n".join(ctx_parts) + "\n" if ctx_parts else ""
        chunks.append({"text": f"{prefix}{heading_label}\n{content}",
                       "heading": heading_label.strip("# "), "source": source})
    for line in text.strip().splitlines():
        if line.startswith("# ") and not line.startswith("## "):
            _flush(); h1_context = line; h2_heading = ""; sub_heading = ""; current_lines = []
        elif line.startswith("## ") and not line.startswith("### "):
            _flush(); h2_heading = line; sub_heading = ""; current_lines = []
        elif line.startswith("### "):
            _flush(); sub_heading = line; current_lines = []
        else:
            current_lines.append(line)
    _flush()
    return chunks

def split_into_chunks(text, source):
    if _is_qa_format(text):
        return _split_qa_chunks(text, source)
    if any(ln.startswith("### ") for ln in text.splitlines()):
        return _split_with_subheadings(text, source)
    chunks, h1_context, current_heading, current_lines = [], "", "", []
    def _flush():
        if not current_lines: return
        content = "\n".join(current_lines).strip()
        if len(content) <= 80: return
        prefix = f"{h1_context}\n" if h1_context else ""
        chunks.append({"text": f"{prefix}{current_heading}\n{content}",
                       "heading": current_heading.strip("# "), "source": source})
    for line in text.strip().splitlines():
        if line.startswith("# ") and not line.startswith("## "):
            _flush(); h1_context = line; current_heading = ""; current_lines = []
        elif line.startswith("## "):
            _flush(); current_heading = line; current_lines = []
        else:
            current_lines.append(line)
    _flush()
    return chunks

print("✅ Функции чанкования готовы")

In [ ]:
# Ячейка 4 — Строим ChromaDB индекс
import warnings
warnings.filterwarnings("ignore")
import chromadb
from chromadb.utils import embedding_functions

E5_DOC_PREFIX   = "passage: "
E5_QUERY_PREFIX = "query: "

_embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-small",
    device="cpu",
)

_client = chromadb.Client()
try: _client.delete_collection("fkn_rag")
except: pass
_collection = _client.create_collection(
    name="fkn_rag",
    embedding_function=_embed_fn,
    metadata={"hnsw:space": "cosine"},
)

all_chunks = []
for path in sorted(DOCS_DIR.glob("*.md")):
    stem_lower = path.stem.lower()
    if any(p in stem_lower for p in _DOCS_EXCLUDE):
        print(f"  ⏭  {path.stem} — пропускаю (тестовый файл)")
        continue
    text   = path.read_text(encoding="utf-8")
    chunks = split_into_chunks(text, path.stem)
    all_chunks.extend(chunks)
    print(f"  📄 {path.stem}: {len(chunks)} чанков")

_collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    documents=[E5_DOC_PREFIX + c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"], "heading": c["heading"]} for c in all_chunks],
)
print(f"\n✅ Индекс готов: {len(all_chunks)} чанков")

In [ ]:
# Ячейка 5 — Retrieval + Generation
from groq import Groq

_SLANG = [
    (r"\bпми\b",          "прикладная математика и информатика"),
    (r"\bпрад\b",         "прикладной анализ данных"),
    (r"\bпад\b",          "прикладной анализ данных"),
    (r"\bэад\b",          "экономика и анализ данных"),
    (r"\bкнад\b",         "компьютерные науки и анализ данных"),
    (r"\bпрогинж\b",      "программная инженерия"),
    (r"\bдрип\b",         "дизайн и разработка информационных продуктов"),
    (r"\bинфу\b",         "информатику"),
    (r"\bинфа\b",         "информатика"),
    (r"\bфизр[ауе]\b",    "физику"),
    (r"\bкмс\b",          "кандидат в мастера спорта"),
    (r"\bмшп\b",          "московская школа программистов"),
    (r"\bяндекс\s+лице[йя]\b", "лицей академии яндекса"),
    (r"\bминималк[иуа]\b",     "минимальные баллы"),
    (r"\bдопбалл\w*\b",        "дополнительные баллы"),
]

SYSTEM_PROMPT = """Ты — помощник по поступлению на ФКН НИУ ВШЭ.
Используй ТОЛЬКО информацию из блоков [источник / раздел] в контексте.
Начинай сразу с сути. Факты с числами выделяй жирным. Максимум 6 предложений.
Если контекст не содержит ответа — скажи об этом."""

def rewrite(q):
    q = q.lower()
    for pat, rep in _SLANG:
        q = re.sub(pat, rep, q, flags=re.IGNORECASE)
    return q

def retrieve(query):
    rw  = rewrite(query)
    res = _collection.query(query_texts=[E5_QUERY_PREFIX + rw], n_results=RAG_TOP_K)
    parts = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        clean = doc[len(E5_DOC_PREFIX):] if doc.startswith(E5_DOC_PREFIX) else doc
        parts.append(f"[{meta['source']} / {meta['heading']}]\n{clean}")
    return "\n\n---\n\n".join(parts)

def generate(question, context):
    client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"=== КОНТЕКСТ ===\n{context}\n=== КОНЕЦ ===\n\n"
            f"Вопрос: {question}\n\nДай чёткий ответ строго по контексту."
        )},
    ]
    resp = client.chat.completions.create(
        model=GROQ_MODEL, messages=messages, max_tokens=500, temperature=0.3
    )
    return resp.choices[0].message.content.strip()

print("✅ RAG-функции готовы")

In [ ]:
# Ячейка 6 — Метрики
from rouge_score import rouge_scorer as _rs

_STOP = {"а","в","и","к","на","не","но","о","от","по","с","у","я","вы",
         "мы","он","она","оно","они","это","то","как","что","где","когда",
         "кто","чем","ли","есть","будет","можно","нужно","нет","да","для",
         "при","из","до","за","со","или","если","же","ещё","уже","тоже","только"}

def tokens(text):
    text = re.sub(r"[^а-яёa-z0-9\s]", " ", text.lower())
    return {t for t in text.split() if t and t not in _STOP and len(t) > 1}

def ctx_recall(ref, context):
    rt = tokens(ref)
    return len(rt & tokens(context)) / len(rt) if rt else 0.0

def rouge1(pred, ref):
    s = _rs.RougeScorer(["rouge1"], use_stemmer=False)
    return s.score(ref, pred)["rouge1"].fmeasure

def rougeL(pred, ref):
    s = _rs.RougeScorer(["rougeL"], use_stemmer=False)
    return s.score(ref, pred)["rougeL"].fmeasure

print("✅ Метрики готовы")

In [ ]:
# Ячейка 7 — Парсинг тест-файла
def parse_qa(path):
    text   = Path(path).read_text(encoding="utf-8")
    pairs  = []
    category = q_id = q_text = a_text = ""
    for line in text.splitlines():
        line = line.rstrip()
        if line.startswith("## Категория"):
            category = line.lstrip("# ").strip()
        elif re.match(r'^\*\*Q\d+\*\*$', line):
            q_id = line.strip("*"); q_text = a_text = ""
        elif line.startswith("> ") and q_id and not q_text:
            q_text = line[2:].strip()
        elif line.startswith("**A:**") and q_id:
            a_text = line[6:].strip()
        elif a_text and q_id and line.strip() and not line.startswith(("---","**Q","## ","# ")):
            a_text += " " + line.strip()
        if q_text and a_text and q_id:
            pairs.append({"id": q_id, "category": category,
                          "question": q_text, "answer": a_text})
            q_id = q_text = a_text = ""
    return pairs

qa_pairs = parse_qa(QA_FILE)
if LIMIT:
    qa_pairs = qa_pairs[:LIMIT]
print(f"✅ Загружено {len(qa_pairs)} тест-пар из {QA_FILE.name}")

In [ ]:
# Ячейка 8 — Прогон
import json
from collections import defaultdict

results   = []
cat_stats = defaultdict(lambda: {"total":0, "ctx_hit":0, "ans_hit":0})

print(f"{'ID':6} {'ctx':5} {'R1':5}  вопрос")
print("-" * 65)

for item in qa_pairs:
    qid      = item["id"]
    question = item["question"]
    ref      = item["answer"]
    cat      = item["category"]

    context = retrieve(question)
    c_rec   = ctx_recall(ref, context)
    ctx_hit = c_rec >= 0.5

    predicted = r1 = rl = ""
    if WITH_LLM:
        try:
            predicted = generate(question, context)
            r1 = rouge1(predicted, ref)
            rl = rougeL(predicted, ref)
        except Exception as e:
            predicted = f"[Ошибка: {e}]"
            r1 = rl = 0.0

    ans_hit = (r1 >= 0.3) if predicted else None
    cat_stats[cat]["total"] += 1
    if ctx_hit: cat_stats[cat]["ctx_hit"] += 1
    if ans_hit: cat_stats[cat]["ans_hit"] += 1

    results.append({
        "id": qid, "category": cat, "question": question,
        "ref_answer": ref, "predicted": predicted,
        "context_recall": round(c_rec, 3), "ctx_hit": ctx_hit,
        "rouge1": round(float(r1), 3) if r1 else 0,
        "rougeL": round(float(rl), 3) if rl else 0,
        "ans_hit": ans_hit,
    })

    ctx_icon = "✅" if ctx_hit else "❌"
    r1_str   = f"{r1:.2f}" if r1 else "—   "
    print(f"  {ctx_icon} {qid:5} {c_rec:.2f}  {r1_str}  {question[:50]}")

print("\n✅ Прогон завершён")

In [ ]:
# Ячейка 9 — Итоговый отчёт
n        = len(results)
ctx_hits = sum(1 for r in results if r["ctx_hit"])
r1_vals  = [r["rouge1"] for r in results if r["predicted"]]
rl_vals  = [r["rougeL"] for r in results if r["predicted"]]
ans_hits = sum(1 for r in results if r["ans_hit"])
n_pred   = len(r1_vals)

print("=" * 55)
print("  ИТОГОВЫЕ МЕТРИКИ")
print("=" * 55)
print(f"  Вопросов всего:          {n}")
print(f"  Context recall ≥0.5:     {ctx_hits}/{n} = {ctx_hits/n*100:.1f}%")
if n_pred:
    print(f"  Avg ROUGE-1:             {sum(r1_vals)/n_pred:.3f}")
    print(f"  Avg ROUGE-L:             {sum(rl_vals)/n_pred:.3f}")
    print(f"  Answer hit (R1≥0.3):     {ans_hits}/{n_pred} = {ans_hits/n_pred*100:.1f}%")

# Провалы
bad = [r for r in results if not r["ctx_hit"]]
if bad:
    print(f"\n  ❌ Слабый retrieval ({len(bad)} шт.):")
    for r in bad:
        print(f"     {r['id']}  ctx={r['context_recall']}  {r['question'][:60]}")

# По категориям
print(f"\n{'='*55}")
print("  ПО КАТЕГОРИЯМ")
print(f"{'='*55}")
for cat, s in cat_stats.items():
    pct = s["ctx_hit"] / s["total"] * 100
    ans_str = f"  ans={s['ans_hit']}/{s['total']}" if n_pred else ""
    print(f"  {cat[:48]:48s}  ctx={s['ctx_hit']}/{s['total']}={pct:.0f}%{ans_str}")

In [ ]:
# Ячейка 10 — Детальный просмотр ответов
# Измени show_only_bad=False чтобы увидеть все вопросы
show_only_bad = True

to_show = [r for r in results if (not r["ctx_hit"]) or (not r["ans_hit"] and r["predicted"])]
if not show_only_bad:
    to_show = results

for r in to_show:
    ctx_icon = "✅" if r["ctx_hit"] else "❌"
    ans_icon = ("✅" if r["ans_hit"] else "❌") if r["predicted"] else ""
    print(f"\n{'─'*60}")
    print(f"{ctx_icon}{ans_icon} {r['id']}  ctx={r['context_recall']}  R1={r['rouge1']}")
    print(f"❓ {r['question']}")
    print(f"📋 ОЖИДАЕМ: {r['ref_answer'][:200]}")
    if r["predicted"]:
        print(f"🤖 ОТВЕТИЛ: {r['predicted'][:200]}")

In [ ]:
# Ячейка 11 — Сохранение результатов
out_path = Path("/kaggle/working/rag_results.json")
out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ Результаты сохранены: {out_path}")
print("   Скачать можно через Output → rag_results.json")